# Solving the Steady-State RTE in a Sphere using a Radial PINN for $I(r,\mu)$

Here we approximate the **intensity $I$ itself** (not the scalar flux $G$), but in **radial coordinates**: a spherically symmetric medium where $I$ depends only on the radius $r$ and on $\mu = \cos\theta$, the cosine of the angle between the propagation direction and the outward radial direction. This is a genuine dimensionality reduction — **2 inputs $(r,\mu)$** instead of the 5 of a general 3D problem.

> **Geometry note.** Radial coordinates require radial symmetry. The square Case 2 (one hot wall) has none, so this notebook solves a **ball of radius 1 illuminated isotropically on its surface** — a different, spherically symmetric test problem. It is the correct setting for the radial form $I(r,\mu)$; it is *not* the square.

### Equation (spherically symmetric RTE)
$$\mu\,\frac{\partial I}{\partial r} + \frac{1-\mu^2}{r}\,\frac{\partial I}{\partial \mu} + (\kappa+\sigma)\,I = \frac{\sigma}{4\pi}\,G(r), \qquad G(r) = 2\pi\!\int_{-1}^{1} I(r,\mu')\,d\mu'$$
so the in-scattering source is $\frac{\sigma}{4\pi}G = \frac{\sigma}{2}\int_{-1}^{1} I\,d\mu'$. The term $\frac{1-\mu^2}{r}\partial_\mu I$ is the **angular redistribution**: in spherical coordinates the direction rotates relative to $\hat r$ as one moves along a ray. It is the signature of the radial form.

### Singularity at $r=0$
The redistribution term is singular at the centre. As in Case 1 (where we multiplied by $x$), we impose the residual in the form **multiplied by $r$**:
$$R = \mu\,r\,\partial_r I + (1-\mu^2)\,\partial_\mu I + (\kappa+\sigma)\,r\,I - \tfrac{\sigma}{2}\,r\!\int_{-1}^{1} I\,d\mu' = 0.$$
At $r=0$ this reduces to $(1-\mu^2)\partial_\mu I = 0$, i.e. $I$ is isotropic at the centre — the physical regularity condition, enforced automatically.

### Boundary condition
Isotropic illumination on the surface: $I(1,\mu) = 1$ for **incoming** directions $\mu < 0$.

### Exact validation anchors (no external reference needed)
* **Pure absorption** ($\sigma = 0$, $\beta = \kappa$): the solution is analytical *everywhere*,
  $$I(r,\mu) = \exp\!\big(-\beta\, d(r,\mu)\big), \qquad d(r,\mu) = r\mu + \sqrt{1 - r^2(1-\mu^2)}$$
  ($d$ = backward distance to the surface along the ray). This is the default headline case: full-field exact check.
* **Isotropic bath** ($\kappa = 0$, $\omega = 1$, incoming $=1$): the exact solution is $I \equiv 1$ everywhere ($G = 4\pi$). Tests the scattering-integral term. Run as a second check.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. PINN Architecture Definition
MLP approximating $I(r, \mu)$ — 2 inputs.

In [ ]:
class PinnRadial(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=4):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(num_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers += [nn.Linear(hidden_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 2. Angular Quadrature (for $G$)
Gauss-Legendre nodes on $\mu \in [-1, 1]$ for the scattering integral $\int_{-1}^{1} I\, d\mu$ (used only when $\sigma > 0$).

In [ ]:
def init_quadrature(N_mu=16, device='cpu'):
    nodes, weights = np.polynomial.legendre.leggauss(N_mu)
    quad_mu = torch.tensor(nodes, dtype=torch.float32).view(1, -1).to(device)
    quad_w = torch.tensor(weights, dtype=torch.float32).view(1, -1).to(device)
    return quad_mu, quad_w

## 3. Exact Solution (pure absorption) and Geometry
$d(r,\mu)$ = backward distance to the surface; $I_{exact} = e^{-\beta d}$ for $\sigma = 0$.

In [ ]:
def dist_bord(r, mu):
    return r * mu + np.sqrt(np.clip(1.0 - r**2 * (1.0 - mu**2), 0.0, None))

def I_exact_abs(r, mu, beta):
    return np.exp(-beta * dist_bord(r, mu))

## 4. Sampling
Collocation points $(r, \mu)$ in $[0,1]\times[-1,1]$; boundary points at $r = 1$ with incoming directions $\mu < 0$.

In [ ]:
def generer_points_collocation(n_pde):
    r = torch.rand(n_pde, 1)
    mu = torch.rand(n_pde, 1) * 2.0 - 1.0
    return r.float(), mu.float()

def generer_points_bords(n_bords):
    r = torch.ones(n_bords, 1)
    mu = -torch.rand(n_bords, 1)          # mu < 0 : directions entrantes
    return r.float(), mu.float()

## 5. Loss Functions
* `calc_bc_loss`: incoming surface intensity $I(1, \mu<0) = 1$.
* `calc_clp_loss`: the radial RTE residual **multiplied by $r$** (handles the $r=0$ singularity), with the scattering integral evaluated by the angular quadrature.

In [ ]:
def calc_bc_loss(model, r_b, mu_b, g_in=1.0):
    pred = model(torch.cat([r_b, mu_b], dim=1))
    return torch.mean((pred - g_in) ** 2)

def calc_clp_loss(model, r, mu, quad_mu, quad_w, kappa=0.0, sigma=1.0):
    r.requires_grad_(True)
    mu.requires_grad_(True)

    I_pred = model(torch.cat([r, mu], dim=1))
    I_r = torch.autograd.grad(I_pred, r, torch.ones_like(I_pred), create_graph=True)[0]
    I_mu = torch.autograd.grad(I_pred, mu, torch.ones_like(I_pred), create_graph=True)[0]

    # source de diffusion (sigma/2) * integrale de I sur mu, par quadrature
    if sigma > 0.0:
        N = r.shape[0]
        N_q = quad_mu.shape[1]
        r_exp = r.repeat(1, N_q)
        mu_exp = quad_mu.repeat(N, 1)
        inp = torch.stack([r_exp, mu_exp], dim=2).view(-1, 2)
        I_q = model(inp).view(N, N_q)
        integ = torch.sum(I_q * quad_w, dim=1, keepdim=True)   # ∫ I dmu
    else:
        integ = torch.zeros_like(I_pred)

    # residu multiplie par r
    res = (mu * r * I_r + (1.0 - mu**2) * I_mu
           + (kappa + sigma) * r * I_pred
           - 0.5 * sigma * r * integ)
    return torch.mean(res ** 2)

## 6. Initialization
Default: **pure absorption** ($\kappa = 1$, $\sigma = 0$) — the case with a full exact solution. Set $\sigma > 0$ to add scattering (then only the isotropic-bath limit has an exact check).

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

quad_mu, quad_w = init_quadrature(N_mu=16, device=device)

kappa = 1.0
sigma = 0.0          # pur absorption : reference exacte plein champ
beta = kappa + sigma

n_pde = 12000
n_bords = 2000

r_c, mu_c = generer_points_collocation(n_pde)
r_c, mu_c = r_c.to(device), mu_c.to(device)
r_b, mu_b = generer_points_bords(n_bords)
r_b, mu_b = r_b.to(device), mu_b.to(device)

modele = PinnRadial(hidden_dim=64, num_layers=4).to(device)

## 7. Model Training (Adam)

In [ ]:
optimizer = optim.Adam(modele.parameters(), lr=2e-3)
epochs = 3000

for epoch in range(epochs):
    optimizer.zero_grad()
    loss_bc = calc_bc_loss(modele, r_b, mu_b)
    loss_clp = calc_clp_loss(modele, r_c, mu_c, quad_mu, quad_w, kappa, sigma)
    loss = 10.0 * loss_bc + loss_clp
    loss.backward()
    optimizer.step()
    if epoch % 200 == 0:
        print(f"Epoque {epoch:04d} | Loss {loss.item():.2e} | CLP {loss_clp.item():.2e} | BC {loss_bc.item():.2e}")

## 8. Model Training (L-BFGS)

In [ ]:
def closure():
    optimizer_lbfgs.zero_grad()
    loss_bc = calc_bc_loss(modele, r_b, mu_b)
    loss_clp = calc_clp_loss(modele, r_c, mu_c, quad_mu, quad_w, kappa, sigma)
    loss = 10.0 * loss_bc + loss_clp
    loss.backward()
    return loss

optimizer_lbfgs = optim.LBFGS(modele.parameters(), line_search_fn="strong_wolfe", max_iter=20)
for epoch in range(500):
    loss = optimizer_lbfgs.step(closure)
    if epoch % 20 == 0:
        print(f"Epoque LBFGS {epoch:03d} | Loss {loss.item():.2e}")

## 9. Results and Validation vs the Exact Solution
For $\sigma = 0$ we compare the full field $I(r,\mu)$ to $e^{-\beta d(r,\mu)}$.

In [ ]:
r_vals = np.linspace(0.0, 1.0, 200)
mu_vals = np.linspace(-1.0, 1.0, 200)
R, MU = np.meshgrid(r_vals, mu_vals, indexing='ij')
grid = torch.tensor(np.stack([R.ravel(), MU.ravel()], 1), dtype=torch.float32).to(device)
with torch.no_grad():
    I_pinn = modele(grid).cpu().numpy().reshape(R.shape)

if sigma == 0.0:
    I_ref = I_exact_abs(R, MU, beta)
    err = np.abs(I_pinn - I_ref)
    print(f"erreur max |I_PINN - I_exact| = {err.max():.3e}")
    print(f"erreur L2 relative            = {np.linalg.norm(I_pinn-I_ref)/np.linalg.norm(I_ref)*100:.2f} %")

    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    for a, Z, t in [(ax[0], I_pinn, "PINN"), (ax[1], I_ref, "Exact")]:
        im = a.pcolormesh(R, MU, Z, cmap='jet', shading='auto', vmin=0, vmax=1)
        a.set_title(f"$I(r,\\mu)$ — {t}"); a.set_xlabel("r"); a.set_ylabel(r"$\mu$")
        fig.colorbar(im, ax=a)
    im = ax[2].pcolormesh(R, MU, err, cmap='inferno', shading='auto')
    ax[2].set_title("|PINN - Exact|"); ax[2].set_xlabel("r"); ax[2].set_ylabel(r"$\mu$")
    fig.colorbar(im, ax=ax[2])
    plt.tight_layout(); plt.savefig("sphere_radial_I_field.png", dpi=150); plt.show()

    # profils le long de mu = -1 (rayon entrant) et mu = +1 (rayon sortant)
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    for a, mu0, lab in [(ax[0], -1.0, "-1 (entrant)"), (ax[1], 1.0, "+1 (sortant)")]:
        inp = torch.tensor(np.stack([r_vals, np.full_like(r_vals, mu0)], 1), dtype=torch.float32).to(device)
        with torch.no_grad():
            Ip = modele(inp).cpu().numpy().flatten()
        a.plot(r_vals, Ip, 'r-', lw=2, label="PINN")
        a.plot(r_vals, I_exact_abs(r_vals, np.full_like(r_vals, mu0), beta), 'k--', lw=1.5, label="exact")
        a.set_title(f"$I(r, \\mu={mu0:+.0f})$  [{lab}]"); a.set_xlabel("r"); a.grid(True); a.legend()
    plt.tight_layout(); plt.savefig("sphere_radial_I_profiles.png", dpi=150); plt.show()
else:
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.pcolormesh(R, MU, I_pinn, cmap='jet', shading='auto')
    ax.set_title(f"$I(r,\\mu)$ PINN (kappa={kappa}, sigma={sigma})")
    ax.set_xlabel("r"); ax.set_ylabel(r"$\mu$"); fig.colorbar(im, ax=ax)
    plt.tight_layout(); plt.savefig("sphere_radial_I_field.png", dpi=150); plt.show()
    print("sigma > 0 : pas de solution exacte plein champ ; utiliser l'ancre du bain isotrope (cellule suivante).")

## 10. Second Exact Anchor: the Isotropic Bath
Retrain with $\kappa = 0$, $\sigma = 1$, incoming $= 1$: the exact solution is $I \equiv 1$ everywhere. This validates the scattering-integral term (which is inactive in the pure-absorption case).

In [ ]:
kappa_b, sigma_b = 0.0, 1.0
modele_b = PinnRadial(hidden_dim=64, num_layers=4).to(device)
opt_b = optim.Adam(modele_b.parameters(), lr=2e-3)
for epoch in range(2000):
    opt_b.zero_grad()
    lbc = calc_bc_loss(modele_b, r_b, mu_b)
    lclp = calc_clp_loss(modele_b, r_c, mu_c, quad_mu, quad_w, kappa_b, sigma_b)
    (10.0 * lbc + lclp).backward()
    opt_b.step()

with torch.no_grad():
    I_bath = modele_b(grid).cpu().numpy()
print(f"bain isotrope : I doit valoir 1 partout")
print(f"  min = {I_bath.min():.4f}, max = {I_bath.max():.4f}, ecart max a 1 = {np.abs(I_bath-1).max():.3e}")

## 11. Notes

* **What this validates.** The pure-absorption case checks the radial transport ($\mu\,r\,\partial_r I$), the angular redistribution ($(1-\mu^2)\partial_\mu I$) and the $r=0$ singularity handling — against a full-field exact solution. The isotropic bath checks the scattering integral. Together they cover the whole radial operator with **zero external reference**.
* **Adding scattering ($\sigma>0$).** The mixed absorbing+scattering ball has no closed form; a 1D spherical $S_N$ or Monte-Carlo reference would be needed to validate it quantitatively.
* **Link to the inverse problem.** Same 2-input, radial machinery as Case 1 but in a genuinely reduced 2D setting — a clean testbed for identifying $\kappa$ or $\sigma$ from $I$ or $G$ observations.